# Productionized Churn Prediction System

This notebook teaches the end-to-end workflow behind the churn prediction system: data validation, leakage prevention, feature engineering, ordinal-aware model selection, explainability, and stakeholder-friendly retention actions.

## 1. Setup and Imports

We load the project modules rather than rewriting logic in the notebook. That keeps the notebook fully aligned with the production training and inference code.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from app.core.config import get_settings
from src.data.load_data import load_train_data, load_test_data
from src.data.validate_data import validate_training_frame
from src.data.preprocess import prepare_training_matrices
from src.pipelines.training_pipeline import run_training_pipeline
settings = get_settings()
train_df = load_train_data()
test_df = load_test_data()
train_df.head()

## 2. Dataset Overview

We inspect shape, schema, and target behavior first. This is also where we verify whether the target should be treated as binary, multiclass, ordinal, or continuous.

In [ ]:
print(train_df.shape)
print(train_df.dtypes)
train_df['churn_risk_score'].value_counts().sort_index()

## 3. Validation Report

Production systems should fail early when data quality drifts. We validate required columns, duplicate IDs, and missing target values before training.

In [ ]:
report = validate_training_frame(train_df, settings.target_column)
report.to_dict()

## 4. Missingness and Leakage Review

This project intentionally removes customer identifiers and personally identifying fields from the training matrix so the model learns behavior rather than memorizing people.

In [ ]:
train_df.isna().sum().sort_values(ascending=False).head(10)

## 5. Feature Engineering Rationale

We derive tenure, time-of-day, activity, complaint, dissatisfaction, value, and loyalty features. The same feature builder is serialized and reused at inference time.

In [ ]:
feature_builder, target_manager, X, y = prepare_training_matrices(train_df)
print(target_manager.metadata())
X.head()

## 6. Full Training Pipeline

The training pipeline benchmarks multiple candidate models, scores them with weighted F1 plus ordinal-aware metrics, saves the winning model bundle, and produces plots and sample outputs.

In [ ]:
summary = run_training_pipeline()
summary

## 7. Saved Metrics and Comparison Table

These files are what the API and Streamlit interface use to present explainability and performance to technical and non-technical audiences.

In [ ]:
comparison = pd.read_csv(settings.comparison_path)
metrics = json.loads(Path(settings.metrics_path).read_text())
comparison

## 8. Explainability and Recommendations

A production churn system should say more than 'this customer is risky'. It should also explain why and what the business should do next.

In [ ]:
from app.services.predictor import PredictorService
predictor = PredictorService()
example_result = predictor.predict_record(train_df.drop(columns=['churn_risk_score']).iloc[0].to_dict())
example_result

## 9. Teaching Takeaways

1. Start with validation, not modeling.
2. Protect against leakage early.
3. Reuse the exact same transformations in training and inference.
4. Optimize for business-relevant metrics and readable outputs.
5. Production value comes from usability, reliability, and actionability, not just a model score.